# Import Libraries

In [1]:
import plotly as plt
import pandas as pd

In [2]:
df = pd.read_csv("../data-visualization/cleaned_master_order_table2.csv")

In [3]:
df

,order_id,order_item_count,unique_sellers,unique_products,total_freight_value,total_item_value,payment_value_total,payment_installments_total,payment_type_count,actual_delivery_days,delivery_delay_days,late_delivery_flag,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,1.0,1.0,1.0,8.72,29.99,38.71,3.0,2.0,8.0,-8.0,False,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,1.0,1.0,1.0,22.76,118.70,141.46,1.0,1.0,13.0,-6.0,False,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,1.0,1.0,1.0,19.22,159.90,179.12,3.0,1.0,9.0,-18.0,False,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,1.0,1.0,1.0,27.20,45.00,72.20,1.0,1.0,13.0,-13.0,False,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,1.0,1.0,1.0,8.72,19.90,28.62,1.0,1.0,2.0,-10.0,False,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95825,9c5dedf39a927c1b2549525ed64a053c,1.0,1.0,1.0,13.08,72.00,85.08,3.0,1.0,8.0,-11.0,False,5.0
95826,63943bddc261676b46f01ca7ac2f7bd8,1.0,1.0,1.0,20.10,174.90,195.00,3.0,1.0,22.0,-2.0,False,4.0
95827,83c1379a015df1e13d02aae0204711ab,1.0,1.0,1.0,65.02,205.99,271.01,5.0,1.0,24.0,-6.0,False,5.0
95828,11c177c8e97725db2631073c19f07b62,2.0,1.0,1.0,81.18,359.98,441.16,4.0,1.0,17.0,-21.0,False,2.0


In [4]:
df.columns

Index(['order_id', 'order_item_count', 'unique_sellers', 'unique_products',
       'total_freight_value', 'total_item_value', 'payment_value_total',
       'payment_installments_total', 'payment_type_count',
       'actual_delivery_days', 'delivery_delay_days', 'late_delivery_flag',
       'review_score'],
      dtype='object')

# Review Score Analysis

Overall customer satisfaction is strong, with an average score of 4.16 and 78.93% of reviews being 4–5 stars. However, the 9.76% share of 1-star reviews represents a meaningful customer-experience risk that should be investigated further.

In [5]:
avg = df['review_score'].mean()
print("Average of review score:", avg)

Average of review score: 4.155661066471877


In [6]:
import plotly.express as px

# Review score distribution
review_dist = (
    df['review_score']
    .value_counts()
    .sort_index()
    .reset_index()
)

review_dist.columns = ['review_score', 'count']

fig = px.bar(
    review_dist,
    x='review_score',
    y='count',
    text='count',
    title='Review Score Distribution',
    labels={
        'review_score': 'Review Score',
        'count': 'Number of Reviews'
    }
)

fig.update_traces(textposition='outside')

fig.update_layout(
    xaxis=dict(
        tickmode='linear',
        dtick=1
    ),
    template='plotly_white'
)

fig.show()

In [7]:
import plotly.express as px

# Calculate percentages
one_star_pct = (df['review_score'] == 1).mean() * 100
four_five_star_pct = df['review_score'].isin([4, 5]).mean() * 100

summary = pd.DataFrame({
    'Category': ['1-Star Reviews', '4–5-Star Reviews'],
    'Percentage': [one_star_pct, four_five_star_pct]
})

fig = px.bar(
    summary,
    x='Category',
    y='Percentage',
    text=summary['Percentage'].round(2).astype(str) + '%',
    title='Customer Satisfaction Distribution',
    labels={'Percentage': 'Percentage of Reviews'}
)

fig.update_traces(textposition='outside')

fig.update_layout(
    yaxis=dict(range=[0, 100]),
    template='plotly_white'
)

fig.show()

# Delivery Impact on Reviews
Late delivery is a major customer-experience risk. While only 6.66% of orders are late, 53.78% of these receive 1-star reviews compared with 6.62% for on-time orders.  
Customer satisfaction deteriorates sharply as delivery delays increase. Even a 1–2 day delay nearly triples the 1-star review rate, while delays of 6+ days result in approximately 68–70% 1-star reviews. This makes delivery delay a critical customer-experience risk.

In [8]:
summary = df.groupby('late_delivery_flag').agg(
    avg_review_score=('review_score', 'mean'),
    one_star_pct=('review_score', lambda x: (x == 1).mean() * 100),
    four_five_star_pct=('review_score', lambda x: x.isin([4, 5]).mean() * 100)
).reset_index()

summary['delivery_status'] = summary['late_delivery_flag'].map({
    False: 'On-time',
    True: 'Late'
})

print(summary[['delivery_status',
               'avg_review_score',
               'one_star_pct',
               'four_five_star_pct']])

  delivery_status  avg_review_score  one_star_pct  four_five_star_pct
0         On-time          4.290135      6.622842           82.652491
1            Late          2.270918     53.776246           26.715763


In [9]:
fig = px.bar(
    x=['On-time', 'Late'],
    y=[
        (df['late_delivery_flag'] == 0).mean() * 100,
        (df['late_delivery_flag'] == 1).mean() * 100
    ],
    text=[
        f"{(df['late_delivery_flag'] == 0).mean() * 100:.2f}%",
        f"{(df['late_delivery_flag'] == 1).mean() * 100:.2f}%"
    ],
    title='% of Orders Delivered On-time vs Late',
    labels={
        'x': 'Delivery Status',
        'y': 'Percentage of Orders'
    }
)

fig.update_traces(textposition='outside')
fig.update_layout(template='plotly_white', yaxis_range=[0, 100])

fig.show()

In [10]:
df['delay_bucket'] = pd.cut(
    df['delivery_delay_days'],
    bins=[-float('inf'), 0, 2, 5, 10, float('inf')],
    labels=[
        'On-time / ≤0 days',
        '1–2 days late',
        '3–5 days late',
        '6–10 days late',
        '>10 days late'
    ]
)

In [11]:
bucket_summary = (
    df.groupby('delay_bucket', observed=False)
      .agg(
          avg_review_score=('review_score', 'mean'),
          one_star_pct=('review_score', lambda x: (x == 1).mean() * 100),
          four_five_star_pct=('review_score', lambda x: x.isin([4, 5]).mean() * 100),
          orders=('review_score', 'count')
      )
      .reset_index()
)

print(bucket_summary.round(2))

        delay_bucket  avg_review_score  one_star_pct  four_five_star_pct  \
0  On-time / ≤0 days              4.29          6.62               82.65   
1      1–2 days late              3.51         18.73               59.29   
2      3–5 days late              2.47         48.39               31.48   
3     6–10 days late              1.77         67.87               13.83   
4      >10 days late              1.71         69.50               12.09   

   orders  
0   89448  
1    1356  
2    1366  
3    1634  
4    2026  


In [12]:
import plotly.express as px

fig = px.line(
    bucket_summary,
    x='delay_bucket',
    y='avg_review_score',
    markers=True,
    title='Delivery Delay -> Average Review Score',
    labels={
        'delay_bucket': 'Delivery Delay',
        'avg_review_score': 'Average Review Score'
    }
)

fig.update_layout(
    template='plotly_white',
    yaxis_range=[0, 5]
)

fig.show()

# Order complexity

## Impact of Number of Sellers
Multi-seller orders show substantially lower review scores despite having lower observed late-delivery rates. Therefore, the relationship cannot be explained by delivery delays alone. However, the multi-seller groups are much smaller, especially 3+ sellers, so this result should be treated as an association rather than a causal effect and investigated further.

In [13]:
import pandas as pd

# Create seller buckets
df['seller_bucket'] = pd.cut(
    df['unique_sellers'],
    bins=[0, 1, 2, float('inf')],
    labels=['1 seller', '2 sellers', '3+ sellers']
)

# Calculate metrics
seller_summary = (
    df.groupby('seller_bucket', observed=False)
      .agg(
          orders=('order_id', 'count'),
          late_delivery_pct=('late_delivery_flag', lambda x: (x == 1).mean() * 100),
          avg_review_score=('review_score', 'mean'),
          one_star_pct=('review_score', lambda x: (x == 1).mean() * 100),
          four_five_star_pct=('review_score', lambda x: x.isin([4, 5]).mean() * 100)
      )
      .reset_index()
)

# Round for readability
seller_summary = seller_summary.round({
    'late_delivery_pct': 2,
    'avg_review_score': 2,
    'one_star_pct': 2,
    'four_five_star_pct': 2
})

print(seller_summary)

  seller_bucket  orders  late_delivery_pct  avg_review_score  one_star_pct  \
0      1 seller   94569               6.74              4.17          9.42   
1     2 sellers    1202               1.00              2.89         34.78   
2    3+ sellers      59               0.00              2.24         52.54   

   four_five_star_pct  
0               79.45  
1               40.18  
2               25.42  


## Impact of Order Size
Larger orders are associated with lower customer satisfaction, despite not having higher late-delivery rates. The 1-star rate increases from 8.50% for single-item orders to 31.69% for 6+ item orders, suggesting that factors beyond delivery delays may affect satisfaction.

In [14]:
# Create order item count buckets
df['item_count_bucket'] = pd.cut(
    df['order_item_count'],
    bins=[0, 1, 3, 5, float('inf')],
    labels=['1 item', '2–3 items', '4–5 items', '6+ items']
)

# Calculate metrics
item_summary = (
    df.groupby('item_count_bucket', observed=False)
      .agg(
          orders=('order_id', 'count'),
          late_delivery_pct=(
              'late_delivery_flag',
              lambda x: (x == 1).mean() * 100
          ),
          avg_review_score=('review_score', 'mean'),
          one_star_pct=(
              'review_score',
              lambda x: (x == 1).mean() * 100
          ),
          four_five_star_pct=(
              'review_score',
              lambda x: x.isin([4, 5]).mean() * 100
          )
      )
      .reset_index()
)

# Round values
item_summary = item_summary.round({
    'late_delivery_pct': 2,
    'avg_review_score': 2,
    'one_star_pct': 2,
    'four_five_star_pct': 2
})

item_summary

,item_count_bucket,orders,late_delivery_pct,avg_review_score,one_star_pct,four_five_star_pct
0,1 item,86301,6.82,4.21,8.50,80.60
1,2–3 items,8606,5.18,3.66,20.47,64.48
2,4–5 items,680,4.41,3.40,26.76,57.94
3,6+ items,243,6.58,3.25,31.69,54.73


# Impact of Unique Products
More distinct products appear to be associated with lower satisfaction, but the data does not show increased late-delivery risk. Therefore, the satisfaction decline may be related to order complexity or product-mix issues rather than delivery delays alone.

In [15]:
# Create unique product buckets
df['product_count_bucket'] = pd.cut(
    df['unique_products'],
    bins=[0, 1, 3, 5, float('inf')],
    labels=['1 product', '2–3 products', '4–5 products', '6+ products']
)

# Calculate the five metrics
product_summary = (
    df.groupby('product_count_bucket', observed=False)
      .agg(
          orders=('order_id', 'count'),
          late_delivery_pct=(
              'late_delivery_flag',
              lambda x: (x == 1).mean() * 100
          ),
          avg_review_score=('review_score', 'mean'),
          one_star_pct=(
              'review_score',
              lambda x: (x == 1).mean() * 100
          ),
          four_five_star_pct=(
              'review_score',
              lambda x: x.isin([4, 5]).mean() * 100
          )
      )
      .reset_index()
)

# Round values
product_summary = product_summary.round({
    'late_delivery_pct': 2,
    'avg_review_score': 2,
    'one_star_pct': 2,
    'four_five_star_pct': 2
})

product_summary

,product_count_bucket,orders,late_delivery_pct,avg_review_score,one_star_pct,four_five_star_pct
0,1 product,92670,6.76,4.18,9.20,79.74
1,2–3 products,3071,3.74,3.38,26.08,55.36
2,4–5 products,75,0.00,3.24,30.67,53.33
3,6+ products,14,0.00,2.86,42.86,35.71


# Freight & Order Value

## Freight Cost vs Satisfaction
Higher freight costs are associated with lower customer satisfaction. The high-freight quartile has an average review score of 3.94 versus 4.31 for the low-freight quartile, while the 1-star rate more than doubles from 6.61% to 14.24%. Higher freight is also associated with a higher late-delivery rate, so the relationship should be investigated further.

In [16]:
import pandas as pd

# Create freight quartile buckets
df['freight_bucket'] = pd.qcut(
    df['total_freight_value'],
    q=4,
    labels=[
        'Low freight',
        'Medium-low',
        'Medium-high',
        'High freight'
    ]
)

# Calculate metrics
freight_summary = (
    df.groupby('freight_bucket', observed=False)
      .agg(
          orders=('order_id', 'count'),
          avg_freight=('total_freight_value', 'mean'),
          avg_review_score=('review_score', 'mean'),
          one_star_pct=(
              'review_score',
              lambda x: (x == 1).mean() * 100
          ),
          four_five_star_pct=(
              'review_score',
              lambda x: x.isin([4, 5]).mean() * 100
          ),
          late_delivery_pct=(
              'late_delivery_flag',
              lambda x: (x == 1).mean() * 100
          )
      )
      .reset_index()
)

# Round for readability
freight_summary = freight_summary.round({
    'avg_freight': 2,
    'avg_review_score': 2,
    'one_star_pct': 2,
    'four_five_star_pct': 2,
    'late_delivery_pct': 2
})

freight_summary

,freight_bucket,orders,avg_freight,avg_review_score,one_star_pct,four_five_star_pct,late_delivery_pct
0,Low freight,23975,10.45,4.31,6.61,83.37,4.35
1,Medium-low,23944,15.46,4.19,8.92,79.79,6.89
2,Medium-high,23958,19.77,4.18,9.29,79.53,7.59
3,High freight,23953,45.38,3.94,14.24,73.01,7.81


## Frieght ratio
Freight burden has only a weak association with customer satisfaction. The average review score declines slightly from 4.20 to 4.11 across freight-ratio quartiles, while late-delivery rates remain broadly stable. Therefore, freight burden does not appear to be a major standalone driver of dissatisfaction.

In [17]:
import pandas as pd

# Calculate freight ratio
df['freight_ratio'] = (
    df['total_freight_value'] / df['total_item_value']
)

# Handle invalid values
df['freight_ratio'] = df['freight_ratio'].replace(
    [float('inf'), -float('inf')],
    pd.NA
)

# Create quantile buckets
df['freight_ratio_bucket'] = pd.qcut(
    df['freight_ratio'],
    q=4,
    labels=[
        'Low ratio',
        'Medium-low',
        'Medium-high',
        'High ratio'
    ]
)

# Calculate metrics
freight_ratio_summary = (
    df.groupby('freight_ratio_bucket', observed=False)
      .agg(
          orders=('order_id', 'count'),
          avg_freight_ratio=('freight_ratio', 'mean'),
          avg_review_score=('review_score', 'mean'),
          one_star_pct=(
              'review_score',
              lambda x: (x == 1).mean() * 100
          ),
          four_five_star_pct=(
              'review_score',
              lambda x: x.isin([4, 5]).mean() * 100
          ),
          late_delivery_pct=(
              'late_delivery_flag',
              lambda x: (x == 1).mean() * 100
          )
      )
      .reset_index()
)

# Round for readability
freight_ratio_summary = freight_ratio_summary.round({
    'avg_freight_ratio': 3,
    'avg_review_score': 2,
    'one_star_pct': 2,
    'four_five_star_pct': 2,
    'late_delivery_pct': 2
})

freight_ratio_summary

,freight_ratio_bucket,orders,avg_freight_ratio,avg_review_score,one_star_pct,four_five_star_pct,late_delivery_pct
0,Low ratio,23959,0.084,4.20,9.57,80.35,6.65
1,Medium-low,23966,0.176,4.16,9.39,79.32,6.45
2,Medium-high,23949,0.292,4.14,9.99,78.56,6.66
3,High ratio,23956,0.682,4.11,10.10,77.47,6.88
